# DataFog PII-NER v1 — Full Training

Train the complete model on all 360K+ examples from AI4Privacy, Nemotron-PII, and Gretel datasets.

**Compute requirements:**
- GPU: A100 (40GB) recommended, T4 (16GB) possible with smaller batch size
- Training time: ~4-6 hours on A100, ~12-18 hours on T4
- Disk: ~5GB for datasets + checkpoints

**Instructions:**
1. Runtime → Change runtime type → **A100 GPU** (or T4)
2. Set your WandB API key in the cell below
3. Click **Run All**

## 1. Setup

In [ ]:
import os, sys

# Clone or force-update to latest
if not os.path.exists("/content/datafog-labs"):
    !git clone https://github.com/DataFog/datafog-labs.git /content/datafog-labs
else:
    !cd /content/datafog-labs && git fetch origin && git reset --hard origin/main

!pip install -e "/content/datafog-labs/pii-ner-v1[dev]" -q

sys.path.insert(0, "/content/datafog-labs/pii-ner-v1/src")

import datafog_pii_ner
print(f"datafog_pii_ner loaded from: {datafog_pii_ner.__file__}")
!cd /content/datafog-labs && git log -1 --oneline

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    try:
        props = torch.cuda.get_device_properties(0)
        mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
        print(f"Memory: {mem / 1e9:.1f} GB")
    except Exception:
        print("Memory: (could not detect)")
else:
    raise RuntimeError("No GPU found — full training requires a GPU")

In [ ]:
# WandB login
import wandb
wandb.login()  # Will prompt for API key if not set

## 2. Configuration

In [ ]:
# Training configuration
# Adjust batch sizes based on your GPU memory:
#   A100 (40GB): batch_size=32, grad_accum=1 -> effective=32
#   T4 (16GB):   batch_size=8,  grad_accum=4 -> effective=32

CONFIG = {
    # Model
    "backbone": "microsoft/deberta-v3-xsmall",
    "max_seq_len": 256,
    "max_char_len": 20,
    "dropout": 0.1,
    
    # Training
    "epochs": 10,
    "batch_size": 32,
    "gradient_accumulation_steps": 1,
    "lr_backbone": 2e-5,
    "lr_head": 1e-3,
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    
    # Data
    "val_ratio": 0.1,
    "test_ratio": 0.1,
    "seed": 42,
    
    # Output
    "output_dir": "/content/pii_ner_v1_output",
    "run_name": "pii-ner-v1-full",
}

# Auto-adjust for GPU type
if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_name = torch.cuda.get_device_name(0)
    if gpu_mem < 20:
        print(f"GPU: {gpu_name} ({gpu_mem:.0f}GB) — adjusting for T4")
        CONFIG["batch_size"] = 8
        CONFIG["gradient_accumulation_steps"] = 4
        # T4 doesn't support bf16, use fp16
        CONFIG["fp16"] = True
        CONFIG["bf16"] = False
    else:
        print(f"GPU: {gpu_name} ({gpu_mem:.0f}GB) — using bf16 (no grad scaler needed)")
        # A100/H100 support bf16 natively — avoids FP16 gradient scaler
        # issues with the CRF layer
        CONFIG["fp16"] = False
        CONFIG["bf16"] = True

effective_batch = CONFIG["batch_size"] * CONFIG["gradient_accumulation_steps"]
print(f"Effective batch size: {effective_batch}")

## 2.5 Quick Validation (no data download)

Run a few training steps on synthetic data to verify the optimizer + mixed precision pipeline works before committing to the full data download.

In [ ]:
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, TrainingArguments
from datafog_pii_ner.data.collator import PiiDataCollator
from datafog_pii_ner.data.label_schema import NUM_LABELS
from datafog_pii_ner.model.pii_model import PiiNerConfig, PiiNerModel
from datafog_pii_ner.training.train import PiiTrainer

print("Creating synthetic data (no download needed)...")
_tokenizer = AutoTokenizer.from_pretrained(CONFIG["backbone"])
_seq_len = 32  # short sequences for speed

# 16 random examples with realistic shapes
_fake_data = {
    "input_ids": np.random.randint(1, 1000, (16, _seq_len)).tolist(),
    "attention_mask": np.ones((16, _seq_len), dtype=int).tolist(),
    "labels": np.random.randint(0, NUM_LABELS, (16, _seq_len)).tolist(),
    "char_ids": np.random.randint(0, 100, (16, _seq_len, CONFIG["max_char_len"])).tolist(),
}
_ds = Dataset.from_dict(_fake_data)
_collator = PiiDataCollator(tokenizer=_tokenizer, max_char_len=CONFIG["max_char_len"])

_config = PiiNerConfig(backbone=CONFIG["backbone"], num_labels=NUM_LABELS)
_model = PiiNerModel(_config)

_args = TrainingArguments(
    output_dir="/tmp/validation_test",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    learning_rate=CONFIG["lr_backbone"],
    fp16=CONFIG.get("fp16", False),
    bf16=CONFIG.get("bf16", False),
    report_to="none",
    logging_steps=1,
    max_steps=3,  # only 3 steps — just testing the pipeline
    remove_unused_columns=False,
    save_strategy="no",
)

_trainer = PiiTrainer(
    model=_model,
    args=_args,
    train_dataset=_ds,
    data_collator=_collator,
    lr_backbone=CONFIG["lr_backbone"],
    lr_head=CONFIG["lr_head"],
)

_result = _trainer.train()
print(f"\nValidation PASSED — 3 training steps completed successfully (loss: {_result.training_loss:.4f})")
print("Optimizer + mixed precision pipeline is working. Safe to proceed with full data download.")

# Clean up to free GPU memory
del _model, _trainer, _ds, _collator, _args, _result
import gc; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 3. Load Data (all 360K+ examples)

In [ ]:
from transformers import AutoTokenizer
from datafog_pii_ner.data.dataset import load_pii_datasets
from datafog_pii_ner.data.label_schema import NUM_LABELS

tokenizer = AutoTokenizer.from_pretrained(CONFIG["backbone"])

print("Loading all datasets (this may take a few minutes)...")
datasets = load_pii_datasets(
    tokenizer=tokenizer,
    max_seq_len=CONFIG["max_seq_len"],
    max_char_len=CONFIG["max_char_len"],
    val_ratio=CONFIG["val_ratio"],
    test_ratio=CONFIG["test_ratio"],
    seed=CONFIG["seed"],
)

print(f"\nDataset sizes:")
print(f"  Train:      {len(datasets['train']):,}")
print(f"  Validation: {len(datasets['validation']):,}")
print(f"  Test:       {len(datasets['test']):,}")
print(f"  Total:      {sum(len(datasets[s]) for s in datasets):,}")
print(f"  Labels:     {NUM_LABELS}")

## 4. Initialize Model

In [ ]:
from datafog_pii_ner.model.pii_model import PiiNerConfig, PiiNerModel

config = PiiNerConfig(
    backbone=CONFIG["backbone"],
    num_labels=NUM_LABELS,
    dropout=CONFIG["dropout"],
)
model = PiiNerModel(config)

param_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {param_count:,}")
print(f"Trainable parameters: {trainable_count:,}")

## 5. Train

In [ ]:
from transformers import TrainingArguments
from datafog_pii_ner.data.collator import PiiDataCollator
from datafog_pii_ner.training.metrics import compute_metrics
from datafog_pii_ner.training.train import PiiTrainer

collator = PiiDataCollator(tokenizer=tokenizer, max_char_len=CONFIG["max_char_len"])

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["lr_backbone"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=CONFIG.get("fp16", False),
    bf16=CONFIG.get("bf16", False),
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="overall_f1",
    load_best_model_at_end=True,
    report_to="wandb",
    run_name=CONFIG["run_name"],
    logging_steps=50,
    remove_unused_columns=False,
    dataloader_num_workers=2,
    save_total_limit=3,
)

print(f"Backbone LR: {CONFIG['lr_backbone']}, Head LR: {CONFIG['lr_head']}")
print(f"Mixed precision: {'bf16' if CONFIG.get('bf16') else 'fp16' if CONFIG.get('fp16') else 'none'}")

# PiiTrainer handles differential learning rates internally via create_optimizer().
# The backbone param group uses eps=1.0 to dampen AdamW's adaptive scaling,
# preventing NaN weight updates caused by bias-correction amplification at step 1
# (see smoke_test_walkthrough.md "AdamW NaN on Full Training" section).
trainer = PiiTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["validation"],
    data_collator=collator,
    compute_metrics=compute_metrics,
    lr_backbone=CONFIG["lr_backbone"],
    lr_head=CONFIG["lr_head"],
)

print(f"\nStarting training for {CONFIG['epochs']} epochs...")
train_result = trainer.train()
print(f"\nTraining complete. Final loss: {train_result.training_loss:.4f}")

## 6. Evaluate on Test Set

In [ ]:
print("Evaluating on held-out test set...")
test_results = trainer.evaluate(datasets["test"])

print("=" * 60)
print("TEST SET RESULTS")
print("=" * 60)
print(f"  Overall F1:        {test_results.get('eval_overall_f1', 0):.4f}")
print(f"  Overall Precision: {test_results.get('eval_overall_precision', 0):.4f}")
print(f"  Overall Recall:    {test_results.get('eval_overall_recall', 0):.4f}")
print()

# Tier recalls
for tier in [1, 2, 3, 4]:
    key = f"eval_tier_{tier}_recall"
    if key in test_results:
        target = {1: 0.98, 2: 0.95, 3: 0.90, 4: 0.85}[tier]
        actual = test_results[key]
        status = "PASS" if actual >= target else "FAIL"
        print(f"  Tier {tier} recall: {actual:.4f}  (target >= {target})  [{status}]")

# Per-type F1 (top 20)
print("\nPer-entity F1 (top 20):")
type_metrics = [(k, v) for k, v in test_results.items() if k.startswith("eval_type_") and k.endswith("_f1")]
type_metrics.sort(key=lambda x: x[1], reverse=True)
for k, v in type_metrics[:20]:
    name = k.replace("eval_type_", "").replace("_f1", "")
    print(f"  {name:30s} {v:.4f}")

## 7. Save Model

In [ ]:
save_path = f"{CONFIG['output_dir']}/best_model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

# Also save the config for reproducibility
import json
with open(f"{save_path}/training_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)
print(f"Training config saved to {save_path}/training_config.json")

# Save test results
with open(f"{save_path}/test_results.json", "w") as f:
    json.dump({k: float(v) if hasattr(v, '__float__') else v for k, v in test_results.items()}, f, indent=2)
print(f"Test results saved to {save_path}/test_results.json")

## 8. Download Model (optional)

Run this cell to download the trained model from Colab.

In [ ]:
# Zip and download the model
import shutil
shutil.make_archive("/content/pii-ner-v1-model", "zip", save_path)
print(f"Model archived to /content/pii-ner-v1-model.zip")

try:
    from google.colab import files
    files.download("/content/pii-ner-v1-model.zip")
except ImportError:
    print("Not running in Colab — download manually")